# 07 The real price at which most households are priced out

How much would the real price have to rise before most households with a bill spend over the affordability line?

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]      # repo root, two levels above this folder
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.common.config import load_config
from src.common.pipeline import build

cfg = load_config()
MODE = os.environ.get("DATA_MODE")          # None = use config; "synthetic" = fake
ghs, ies, nfcm = build(MODE, save=False)

C:\Users\mmjmo\Projects\sa-electricity-nonpayment\src\common\pipeline.py:28: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(full, convert_categoricals=False)


**Note.** Two households: bills of 5% and 3% of income. The price doubles, so the bills become 10% and 6%. At a 10% line only the first is above it, so half of the households are.

In [2]:
from src.a_who_pays import affordability, price_multiplier as pm

pool = affordability.paying_households(ies)
mult = pm.multiplier_table(pool, cfg["price_multiplier"]["multipliers"], [0.08, 0.10, 0.12, 0.15])
shares = mult.set_index("multiplier") * 100          # shares of households, in %
shares.round(1)

,share_above_8pct,share_above_10pct,share_above_12pct,share_above_15pct
multiplier,,,,
1.00,24.3,18.4,14.3,10.5
1.25,31.3,24.3,19.4,14.3
1.50,37.9,29.8,24.3,18.4
2.00,48.9,40.1,33.3,26.1
2.50,58.2,48.9,41.6,33.3
3.00,65.5,56.4,48.9,40.1
4.00,75.1,67.6,60.7,51.7


### At what real price do most households cross the line?
Half of the households with a bill spend 10% or more of their income on electricity when the real price is about 2.57 times the 2022/23 level, if households keep using the same amount. At an 8% line the multiple is about 2.05, at 12% about 3.08 and at 15% about 3.85. Today's real price is already roughly 1.25 to 1.3 times the survey level (my rough estimate), which puts about a quarter of households (24%) above the 10% line, against 18% in the survey. Doubling the real price would put about 40% above it.

In [3]:
crossing = pm.crossing_table(pool, [0.08, 0.10, 0.12, 0.15], cfg["price_multiplier"]["elasticities"])
crossing.round(2)

,threshold,elasticity_0.0,elasticity_-0.2,elasticity_-0.5
0,0.08,2.05,2.46,4.21
1,0.10,2.57,3.25,6.58
2,0.12,3.08,4.08,9.48
3,0.15,3.85,5.39,14.81


### What does a demand response change?
If households cut their use when prices rise, their bills rise by less than the price. With an elasticity of -0.2 the real price needed to put half of households above the 10% line rises from 2.57 to 3.25 times, and with -0.5 to 6.58 times. That does not make households better off. Cutting use means less lighting, cooking or refrigeration, so the share above the line understates the harm when demand falls.

In [4]:
cfg_pm = cfg["price_multiplier"]
pm.years_table(crossing, cfg_pm["current_real_multiplier"], cfg_pm["real_growth_rates"]).round(1)

,threshold,multiplier_needed,years_at_3pct,years_at_5pct,years_at_7pct
0,0.1,2.1,16.0,9.7,7.0
1,0.1,2.6,23.5,14.3,10.3
2,0.1,3.1,29.7,18.0,13.0
3,0.2,3.8,37.2,22.6,16.3


**Years to reach the crossing point.** From today's rough real level (1.28 times), reaching the 10% crossing (2.57 times) takes about 10 years at 7% real growth, 14 years at 5% and 24 years at 3%. With an elasticity of -0.2 it takes about 14, 19 and 32 years.

In [5]:
print(round(pm.years_to_reach(2.0, 1.0, 0.07), 2))   # years to double at 7% a year: 10.24

10.24


### Does every tariff increase above inflation create new defaulters?
Not exactly. What matters is the bill as a share of household income, so the right comparison is income growth and not CPI. A real increase pushes households near the line over it: each extra 25% in the real price moves about 6 percentage points of households, roughly 1 million, above the 10% line (18% at 1.0 times, 24% at 1.25 times, 30% at 1.5 times). Being above the line is not the same as defaulting. Households may cut use, rely on free basic units, steal electricity or install solar, and I do not know what share actually stop paying. That is the question for Project C. Tariffs were above CPI in 17 of 18 years, so annual data cannot separate years with and without an above-inflation increase. A better test needs the size of each year's real increase and a year-by-year measure of non-payment, with controls for load shedding, unemployment and municipal governance.

### Limits 

1. The current real multiplier (1.28) is a rough estimate that assumes household incomes tracked CPI since 2022/23.
2. The income distribution is held fixed, so rising or falling real incomes are not modelled.
3. Only households with a bill are counted, and households that already ration electricity understate the burden.
4. The 10% line is a convention, and the elasticities are assumptions, not estimates.
5. This shows the point at which bills become a heavy share of income, not the point at which households stop paying.